In [1]:
import requests

url = "https://api.open-meteo.com/v1/forecast"

response = requests.get(url)

print(response.status_code)


200


In [2]:
import requests

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 22.80,
    "longitude": 86.20,
    "current": "temperature_2m"
}

response = requests.get(url, params=params)

print(response.status_code)

200


In [3]:
data = response.json()

print(data)

{'latitude': 22.811949, 'longitude': 86.15625, 'generationtime_ms': 0.016689300537109375, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 160.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C'}, 'current': {'time': '2026-10-13T04:45', 'interval': 900, 'temperature_2m': 29.9}}


In [4]:
{
    'current': {
        'time': '2026-10-08T10:30',
        'temperature_2m': 28.5
    }
}

{'current': {'time': '2026-10-08T10:30', 'temperature_2m': 28.5}}

In [5]:
data = response.json()

print(data)

{'latitude': 22.811949, 'longitude': 86.15625, 'generationtime_ms': 0.016689300537109375, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 160.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C'}, 'current': {'time': '2026-10-13T04:45', 'interval': 900, 'temperature_2m': 29.9}}


In [6]:
print(data["current"]["time"])
print(data["current"]["temperature_2m"])

2026-10-13T04:45
29.9


In [7]:

import requests

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 22.80,
    "longitude": 86.20,
    "start_date": "2026-09-01",
    "end_date": "2026-09-03",
    "daily": [
        "temperature_2m_mean",
        "precipitation_sum",
        "wind_speed_10m_max"
    ],
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params)

print(response.status_code)

200


In [8]:

data = response.json()

print(data["daily"])

{'time': ['2026-09-01', '2026-09-02', '2026-09-03'], 'temperature_2m_mean': [26.7, 27.0, 27.3], 'precipitation_sum': [42.4, 19.4, 2.1], 'wind_speed_10m_max': [8.8, 7.6, 11.3]}


{'time': ['2026-09-01', '2026-09-02', '2026-09-03'],
 'temperature_2m_mean': [29.1, 28.4, 27.8],
 'precipitation_sum': [2.0, 5.4, 0.0],
 'wind_speed_10m_max': [12.5, 15.2, 10.8]}

In [9]:

import json

with open("weather_raw.json", "w") as file:
    json.dump(data, file, indent=4)

print("Raw weather data saved successfully!")

Raw weather data saved successfully!


In [10]:

with open("weather_raw.json", "r") as file:
    saved_data = json.load(file)

print(saved_data)

{'latitude': 22.811949, 'longitude': 86.15625, 'generationtime_ms': 0.12218952178955078, 'utc_offset_seconds': 19800, 'timezone': 'Asia/Kolkata', 'timezone_abbreviation': 'GMT+5:30', 'elevation': 160.0, 'daily_units': {'time': 'iso8601', 'temperature_2m_mean': '°C', 'precipitation_sum': 'mm', 'wind_speed_10m_max': 'km/h'}, 'daily': {'time': ['2026-09-01', '2026-09-02', '2026-09-03'], 'temperature_2m_mean': [26.7, 27.0, 27.3], 'precipitation_sum': [42.4, 19.4, 2.1], 'wind_speed_10m_max': [8.8, 7.6, 11.3]}}


In [11]:
import requests

def extract_weather(city_name, latitude, longitude):

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": "2026-09-01",
        "end_date": "2026-09-03",
        "daily": [
            "temperature_2m_mean",
            "precipitation_sum",
            "wind_speed_10m_max"
        ],
        "timezone": "Asia/Kolkata"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    return data

In [12]:
weather_data = extract_weather(
    "Jamshedpur",
    22.80,
    86.20
)

print(weather_data["daily"])

{'time': ['2026-09-01', '2026-09-02', '2026-09-03'], 'temperature_2m_mean': [26.7, 27.0, 27.3], 'precipitation_sum': [42.4, 19.4, 2.1], 'wind_speed_10m_max': [8.8, 7.6, 11.3]}


In [13]:
cities = [
    {"city": "Jamshedpur", "latitude": 22.80, "longitude": 86.20},
    {"city": "Delhi", "latitude": 28.61, "longitude": 77.21},
    {"city": "Mumbai", "latitude": 19.08, "longitude": 72.88},
    {"city": "Kolkata", "latitude": 22.57, "longitude": 88.36}
]

print(cities)

[{'city': 'Jamshedpur', 'latitude': 22.8, 'longitude': 86.2}, {'city': 'Delhi', 'latitude': 28.61, 'longitude': 77.21}, {'city': 'Mumbai', 'latitude': 19.08, 'longitude': 72.88}, {'city': 'Kolkata', 'latitude': 22.57, 'longitude': 88.36}]


In [14]:
import json
from pathlib import Path

# Create a folder to store raw data
raw_folder = Path("data/raw")
raw_folder.mkdir(parents=True, exist_ok=True)

# Extract data for each city
for city_info in cities:
    city_name = city_info["city"]
    latitude = city_info["latitude"]
    longitude = city_info["longitude"]

    try:
        weather_data = extract_weather(
            city_name,
            latitude,
            longitude
        )

        # Create a filename for the city
        file_path = raw_folder / f"{city_name.lower()}_weather.json"

        # Save the raw API response
        with open(file_path, "w") as file:
            json.dump(weather_data, file, indent=4)

        print(f"{city_name}: data extracted and saved successfully")

    except Exception as error:
        print(f"{city_name}: extraction failed - {error}")

Jamshedpur: data extracted and saved successfully
Delhi: data extracted and saved successfully
Mumbai: data extracted and saved successfully
Kolkata: data extracted and saved successfully


In [15]:
for file_path in raw_folder.glob("*.json"):
    print(file_path)

data\raw\delhi_weather.json
data\raw\jamshedpur_weather.json
data\raw\kolkata_weather.json
data\raw\mumbai_weather.json


In [16]:
import json
import pandas as pd
from pathlib import Path

In [17]:
weather_rows = []

for file_path in raw_folder.glob("*.json"):

    with open(file_path, "r") as file:
        data = json.load(file)

    city_name = file_path.stem.replace("_weather", "").title()

    daily_data = data["daily"]

    for i in range(len(daily_data["time"])):
        row = {
            "city": city_name,
            "date": daily_data["time"][i],
            "temperature": daily_data["temperature_2m_mean"][i],
            "precipitation": daily_data["precipitation_sum"][i],
            "max_wind_speed": daily_data["wind_speed_10m_max"][i]
        }

        weather_rows.append(row)

weather_df = pd.DataFrame(weather_rows)

print(weather_df)

          city        date  temperature  precipitation  max_wind_speed
0        Delhi  2026-09-01         30.3            3.9            10.4
1        Delhi  2026-09-02         28.0           13.6             9.3
2        Delhi  2026-09-03         28.3            3.5             9.5
3   Jamshedpur  2026-09-01         26.7           42.4             8.8
4   Jamshedpur  2026-09-02         27.0           19.4             7.6
5   Jamshedpur  2026-09-03         27.3            2.1            11.3
6      Kolkata  2026-09-01         27.3           28.2            16.7
7      Kolkata  2026-09-02         27.6           36.3            14.1
8      Kolkata  2026-09-03         27.8           14.2            10.0
9       Mumbai  2026-09-01         27.5            6.1            17.5
10      Mumbai  2026-09-02         26.6           13.8            14.1
11      Mumbai  2026-09-03         27.1            5.6            16.2


In [19]:
print("Shape:", weather_df.shape)
print("\nData types:")
print(weather_df.dtypes)

print("\nMissing values:")
print(weather_df.isnull().sum())

Shape: (12, 5)

Data types:
city                  str
date                  str
temperature       float64
precipitation     float64
max_wind_speed    float64
dtype: object

Missing values:
city              0
date              0
temperature       0
precipitation     0
max_wind_speed    0
dtype: int64


In [18]:
# 1. Display all transformed records
print(weather_df)

# 2. Check unique cities
print("Unique cities:", weather_df["city"].nunique())
print(weather_df["city"].unique())

# 3. Check duplicate records
print("Duplicate rows:", weather_df.duplicated().sum())

# 4. Check missing values
print("\nMissing values:")
print(weather_df.isnull().sum())

# 5. Check data types
print("\nData types:")
print(weather_df.dtypes)

          city        date  temperature  precipitation  max_wind_speed
0        Delhi  2026-09-01         30.3            3.9            10.4
1        Delhi  2026-09-02         28.0           13.6             9.3
2        Delhi  2026-09-03         28.3            3.5             9.5
3   Jamshedpur  2026-09-01         26.7           42.4             8.8
4   Jamshedpur  2026-09-02         27.0           19.4             7.6
5   Jamshedpur  2026-09-03         27.3            2.1            11.3
6      Kolkata  2026-09-01         27.3           28.2            16.7
7      Kolkata  2026-09-02         27.6           36.3            14.1
8      Kolkata  2026-09-03         27.8           14.2            10.0
9       Mumbai  2026-09-01         27.5            6.1            17.5
10      Mumbai  2026-09-02         26.6           13.8            14.1
11      Mumbai  2026-09-03         27.1            5.6            16.2
Unique cities: 4
<StringArray>
['Delhi', 'Jamshedpur', 'Kolkata', 'Mumbai']
L

In [19]:
# 1. Display all transformed records
print(weather_df)


          city        date  temperature  precipitation  max_wind_speed
0        Delhi  2026-09-01         30.3            3.9            10.4
1        Delhi  2026-09-02         28.0           13.6             9.3
2        Delhi  2026-09-03         28.3            3.5             9.5
3   Jamshedpur  2026-09-01         26.7           42.4             8.8
4   Jamshedpur  2026-09-02         27.0           19.4             7.6
5   Jamshedpur  2026-09-03         27.3            2.1            11.3
6      Kolkata  2026-09-01         27.3           28.2            16.7
7      Kolkata  2026-09-02         27.6           36.3            14.1
8      Kolkata  2026-09-03         27.8           14.2            10.0
9       Mumbai  2026-09-01         27.5            6.1            17.5
10      Mumbai  2026-09-02         26.6           13.8            14.1
11      Mumbai  2026-09-03         27.1            5.6            16.2


In [20]:
# 2. Check unique cities
print("Unique cities:", weather_df["city"].nunique())
print(weather_df["city"].unique())

Unique cities: 4
<StringArray>
['Delhi', 'Jamshedpur', 'Kolkata', 'Mumbai']
Length: 4, dtype: str


In [21]:
# 3. Check duplicate records
print("Duplicate rows:", weather_df.duplicated().sum())

Duplicate rows: 0


In [22]:
 #4. Check missing values
print("\nMissing values:")
print(weather_df.isnull().sum())


Missing values:
city              0
date              0
temperature       0
precipitation     0
max_wind_speed    0
dtype: int64


In [23]:
# 5. Check data types
print("\nData types:")
print(weather_df.dtypes)


Data types:
city                  str
date                  str
temperature       float64
precipitation     float64
max_wind_speed    float64
dtype: object


In [26]:
print("Testing notebook")
print(weather_df.shape)

Testing notebook
(12, 5)


In [24]:
print(weather_df.to_string(index=False))

      city       date  temperature  precipitation  max_wind_speed
     Delhi 2026-09-01         30.3            3.9            10.4
     Delhi 2026-09-02         28.0           13.6             9.3
     Delhi 2026-09-03         28.3            3.5             9.5
Jamshedpur 2026-09-01         26.7           42.4             8.8
Jamshedpur 2026-09-02         27.0           19.4             7.6
Jamshedpur 2026-09-03         27.3            2.1            11.3
   Kolkata 2026-09-01         27.3           28.2            16.7
   Kolkata 2026-09-02         27.6           36.3            14.1
   Kolkata 2026-09-03         27.8           14.2            10.0
    Mumbai 2026-09-01         27.5            6.1            17.5
    Mumbai 2026-09-02         26.6           13.8            14.1
    Mumbai 2026-09-03         27.1            5.6            16.2


In [25]:
print("Unique cities:", weather_df["city"].nunique())
print("Duplicate rows:", weather_df.duplicated().sum())
print("Missing values:")
print(weather_df.isnull().sum())

Unique cities: 4
Duplicate rows: 0
Missing values:
city              0
date              0
temperature       0
precipitation     0
max_wind_speed    0
dtype: int64


In [26]:
weather_df["date"] = pd.to_datetime(weather_df["date"])

print(weather_df.dtypes)

city                         str
date              datetime64[us]
temperature              float64
precipitation            float64
max_wind_speed           float64
dtype: object


In [30]:
print(weather_df[["city", "date"]].head())

         city       date
0       Delhi 2026-09-01
1       Delhi 2026-09-02
2       Delhi 2026-09-03
3  Jamshedpur 2026-09-01
4  Jamshedpur 2026-09-02


In [27]:
print(weather_df[["temperature", "precipitation", "max_wind_speed"]].describe())

       temperature  precipitation  max_wind_speed
count    12.000000      12.000000       12.000000
mean     27.625000      15.758333       12.125000
std       0.980839      13.432082        3.421888
min      26.600000       2.100000        7.600000
25%      27.075000       5.175000        9.450000
50%      27.400000      13.700000       10.850000
75%      27.850000      21.600000       14.625000
max      30.300000      42.400000       17.500000


In [28]:
print("Negative precipitation values:",
      (weather_df["precipitation"] < 0).sum())


Negative precipitation values: 0


In [29]:
print("Negative wind speed values:",
      (weather_df["max_wind_speed"] < 0).sum())

Negative wind speed values: 0


In [34]:
print("Missing values:",
      weather_df.isnull().sum().sum())

Missing values: 0


In [35]:
from pathlib import Path

processed_folder = Path("data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)

csv_path = processed_folder / "weather_cleaned.csv"

weather_df.to_csv(csv_path, index=False)

print("Cleaned weather data saved successfully!")
print("File location:", csv_path)

Cleaned weather data saved successfully!
File location: data\processed\weather_cleaned.csv


In [36]:
check_df = pd.read_csv(csv_path)

print("Shape:", check_df.shape)
print(check_df.head())

Shape: (12, 5)
         city        date  temperature  precipitation  max_wind_speed
0       Delhi  2026-09-01         30.3            3.9            10.4
1       Delhi  2026-09-02         28.0           13.6             9.3
2       Delhi  2026-09-03         28.3            3.5             9.5
3  Jamshedpur  2026-09-01         26.7           42.4             8.8
4  Jamshedpur  2026-09-02         27.0           19.4             7.6


In [37]:
%pip install sqlalchemy pyodbc

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [38]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

server = r"Mehnaz-PC\SQLEXPRESS"
database = "WeatherDW"

connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

connection_url = (
    "mssql+pyodbc:///?odbc_connect="
    + quote_plus(connection_string)
)

engine = create_engine(connection_url)

print("Connection engine created successfully!")

Connection engine created successfully!


In [39]:
from sqlalchemy import text

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT DB_NAME() AS CurrentDatabase")
    )

    print(result.scalar())

WeatherDW


In [40]:
# Prepare the DataFrame for SQL Server
weather_to_load = weather_df.rename(columns={
    "city": "City",
    "date": "WeatherDate",
    "temperature": "Temperature",
    "precipitation": "Precipitation",
    "max_wind_speed": "MaxWindSpeed"
}).copy()

# Convert dates to Python date values
weather_to_load["WeatherDate"] = (
    weather_to_load["WeatherDate"].dt.date
)

# Load the records into the staging table
weather_to_load.to_sql(
    name="StgWeather",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

print("Weather data loaded successfully!")
print("Rows loaded:", len(weather_to_load))

Weather data loaded successfully!
Rows loaded: 12


In [30]:

weather_df.head()

,city,date,temperature,precipitation,max_wind_speed
0,Delhi,2026-09-01,30.3,3.9,10.4
1,Delhi,2026-09-02,28.0,13.6,9.3
2,Delhi,2026-09-03,28.3,3.5,9.5
3,Jamshedpur,2026-09-01,26.7,42.4,8.8
4,Jamshedpur,2026-09-02,27.0,19.4,7.6


In [31]:

print(weather_df.shape)


(12, 5)


In [32]:

weather_df.to_csv(
    "weather_cleaned.csv",
    index=False
)

In [33]:

import os
print(os.getcwd())

C:\Users\mehna\Weather_Data_Engineering


In [34]:
weather_df["max_wind_speed"].isna().sum()

np.int64(0)